In [82]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       946Mi       8.6Gi       5.0Mi       3.2Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [84]:
!git clone https://github.com/networkx/networkx.git
%cd networkx

!git checkout networkx-2.8.8

!pip install -e .

fatal: destination path 'networkx' already exists and is not an empty directory.
/content/networkx/networkx/networkx
Previous HEAD position was f4b528e23 Designate 2.6.3 release
HEAD is now at 9256ef670 Designate 2.8.8 release
Obtaining file:///content/networkx/networkx/networkx
  Preparing metadata (setup.py) ... done
  Attempting uninstall: networkx
    Found existing installation: networkx 2.6.3
    Uninstalling networkx-2.6.3:
      Successfully uninstalled networkx-2.6.3
  Running setup.py develop for networkx
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
mapclassify 2.10.0 requires netwo

## Optimization

Replaced Python-based Brandes implementation with a Numba-accelerated version using CSR graph representation.

Why faster:
- Avoids Python loops
- Uses compiled execution via Numba
- Improves memory locality

Trade-offs:
- Higher memory usage (CSR format)
- Less flexible than NetworkX graph objects

In [1]:
import networkx as nx

G = nx.erdos_renyi_graph(n=2000, p=0.01, seed=42)

# Reproducibility

We fix the random seed (`seed=42`) while generating the graph to ensure that the same input graph is used across all runs.

This guarantees that baseline and candidate implementations are evaluated on identical data, making the performance comparison fair and reproducible.

In [2]:
# graph_to_csr
def graph_to_csr(G):
    import numpy as np

    n = len(G)
    indptr = np.zeros(n + 1, dtype=np.int32)
    indices = []

    count = 0
    for i, node in enumerate(G):
        neighbors = list(G.neighbors(node))
        count += len(neighbors)
        indptr[i + 1] = count
        indices.extend(neighbors)

    indices = np.array(indices, dtype=np.int32)
    return indptr, indices

In [3]:
import numba
import numpy as np

@numba.njit(cache=True)
def _brandes_numba_core(indptr, indices, n):
    bc = np.zeros(n, dtype=np.float64)

    for s in range(n):
        # --- initialization ---
        stack = np.empty(n, dtype=np.int32)
        stack_ptr = 0

        queue = np.empty(n, dtype=np.int32)
        q_start = 0
        q_end = 0

        dist = np.full(n, -1, dtype=np.int32)
        sigma = np.zeros(n, dtype=np.float64)

        dist[s] = 0
        sigma[s] = 1.0

        queue[q_end] = s
        q_end += 1

        # --- BFS ---
        while q_start < q_end:
            v = queue[q_start]
            q_start += 1

            stack[stack_ptr] = v
            stack_ptr += 1

            for i in range(indptr[v], indptr[v + 1]):
                w = indices[i]

                if dist[w] < 0:
                    dist[w] = dist[v] + 1
                    queue[q_end] = w
                    q_end += 1

                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]

        # --- dependency accumulation ---
        delta = np.zeros(n, dtype=np.float64)

        for i in range(stack_ptr - 1, -1, -1):
            w = stack[i]

            for j in range(indptr[w], indptr[w + 1]):
                v = indices[j]

                if dist[v] == dist[w] - 1:
                    if sigma[w] != 0:
                        delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])

            if w != s:
                bc[w] += delta[w]

    return bc

In [4]:
indptr, indices = graph_to_csr(G)

def run():
    return _brandes_numba_core(indptr, indices, len(indptr)-1)

In [5]:
def run_normalised():
    raw = run()
    n = len(raw)

    if n <= 2:
        return raw

    scale = 1.0 / ((n - 1) * (n - 2))
    return raw * scale

In [6]:
output = run_normalised()

np.save("candidate_output.npy", output)

In [7]:
!ls

candidate_output.npy  networkx	sample_data


In [8]:
import time
import numpy as np

def benchmark(run, n_warmup=1, n_runs=3):
    for _ in range(n_warmup):
        run()

    times = []
    for _ in range(n_runs):
        start = time.time()
        run()
        times.append(time.time() - start)

    return np.median(times), np.std(times)

median, std = benchmark(run)

print("Candidate median:", median)
print("Std:", std)

Candidate median: 0.8289361000061035
Std: 0.006240013455808857


In [10]:
from google.colab import files
files.download("candidate_output.npy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>